# 01 — Ingest Dokumen

**Tujuan:** mengumpulkan bahan penelitian: memindai PDF & snapshot HTML LPMB yang
sudah tersedia di `data/raw/pdf/` dan `data/raw/web/`, mengekstraksi teksnya, dan
menghasilkan katalog dokumen (§6 `perancangan.md`).

Notebook ini TIDAK mengunduh dari internet — sumber PDF/HTML sudah ditaruh manual
(atau lewat `ingest.download_pdf`/`ingest.snapshot_web_page` di luar notebook ini)
sebelum notebook dijalankan. Snapshot web menulis sidecar `.meta.json` berisi URL
asli, dipakai untuk mengisi `source_url` saat ekstraksi.

**INPUT:** `../data/raw/pdf/*.pdf`, `../data/raw/web/*.html`

**OUTPUT:** `../data/interim/{doc_id}.txt` (teks per halaman), `../data/processed/documents.jsonl`

**Perkiraan runtime:** beberapa detik per 100 halaman PDF; hampir instan per laman web

**Perkiraan biaya API:** Rp 0 — tidak ada panggilan LLM di notebook ini

**Assert di notebook ini:** setiap PDF/HTML di `data/raw/{pdf,web}/` menghasilkan tepat
satu baris di `documents.jsonl`; `doc_id` unik; `sha256` terisi; `local_path` menunjuk
berkas yang ada.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../src")

In [2]:
import pandas as pd

from config import load_config, set_seed

cfg = load_config("../configs/base.yaml")
set_seed(cfg.seed)

## Pindai PDF lokal & hitung sha256

`sha256` dipakai untuk mendeteksi duplikat (dua nama berkas berbeda, isi identik).

In [3]:
import ingest

pdf_paths = ingest.scan_local_pdfs("../data/raw/pdf")
hashes = {p: ingest.sha256_of_file(p) for p in pdf_paths}

dupes = pd.Series(list(hashes.values())).value_counts()
dupes = dupes[dupes > 1]
if len(dupes):
    print("PERINGATAN — sha256 duplikat terdeteksi (kemungkinan berkas sama diunggah dua kali):")
    for h in dupes.index:
        print(" ", h, [p.name for p in hashes if hashes[p] == h])

print(f"{len(pdf_paths)} PDF ditemukan di data/raw/pdf/")

1 PDF ditemukan di data/raw/pdf/


## Metadata dokumen (EDIT bila perlu)

`program`/`year`/`title` ditebak otomatis dari nama berkas. Bila nama berkas kriptik
(mis. `"794KR PPENGMAS PKM DPR 2026"`), override di `metadata_overrides` sebelum
melanjutkan — kunci dict adalah nama berkas persis seperti di `data/raw/pdf/`.

In [4]:
import re


def guess_metadata(filename: str) -> dict:
    stem = re.sub(r"\s*\(\d+\)$", "", Path(filename).stem)  # buang sufiks " (2)" dsb
    year_match = re.search(r"20\d{2}", stem)
    year = int(year_match.group()) if year_match else pd.Timestamp.now().year
    return {"title": stem, "program": "UNKNOWN", "year": year}


from pathlib import Path

# EDIT di sini: kunci = nama berkas persis di data/raw/pdf/, nilai = override
# metadata (title/program/year). Baris yang tidak disebutkan memakai tebakan
# otomatis dari guess_metadata() di atas.
metadata_overrides = {
    "Panduan Usulan Pengmas Internal Universitas Airlangga Tahun 2026 (2).pdf": {
        "title": "Panduan Usulan Pengmas Internal Universitas Airlangga Tahun 2026",
        "program": "Pengmas Internal",
        "year": 2026,
    },
}

In [5]:
rows = []
for p in pdf_paths:
    meta = {**guess_metadata(p.name), **metadata_overrides.get(p.name, {})}
    rows.append({"filename": p.name, "local_path": p, "sha256": hashes[p], **meta})

meta_df = pd.DataFrame(rows)
meta_df

,filename,local_path,sha256,title,program,year
0,Panduan Usulan Pengmas Internal Universitas Ai...,..\data\raw\pdf\Panduan Usulan Pengmas Interna...,16462b0984d92320b8d08cebe2912e11fd57286680dba2...,Panduan Usulan Pengmas Internal Universitas Ai...,Pengmas Internal,2026


## Ekstraksi teks (PyMuPDF + pdfplumber untuk halaman bertabel)

In [6]:
extracted = {}  # local_path -> list[{"page", "text", "method"}]
for row in meta_df.itertuples():
    pages = ingest.extract_text_auto(row.local_path)
    extracted[row.local_path] = pages
    n_table_pages = sum(1 for pg in pages if pg["method"] == "pdfplumber_table")
    print(f"{row.filename}: {len(pages)} halaman, {n_table_pages} lewat pdfplumber (tabel)")

Panduan Usulan Pengmas Internal Universitas Airlangga Tahun 2026 (2).pdf: 99 halaman, 47 lewat pdfplumber (tabel)


## Pindai snapshot web lokal & hitung sha256

Laman diarsipkan lebih dulu ke `data/raw/web/` (mis. lewat `ingest.snapshot_web_page`
di luar notebook ini) sebelum notebook dijalankan — sama seperti PDF di atas, notebook
ini hanya membaca snapshot yang sudah ada, tidak mengunduh apa pun.

In [7]:
html_paths = ingest.scan_local_html("../data/raw/web")
html_hashes = {p: ingest.sha256_of_file(p) for p in html_paths}

print(f"{len(html_paths)} snapshot HTML ditemukan di data/raw/web/")

0 snapshot HTML ditemukan di data/raw/web/


## Metadata laman web (EDIT bila perlu)

`title`/`program`/`year` ditebak dari sidecar `.meta.json` (URL) dan nama berkas;
`source_url` diisi otomatis dari sidecar bila ada, `null` bila snapshot ditaruh manual
tanpa lewat `ingest.snapshot_web_page`. Override di `web_metadata_overrides` bila
tebakan kurang tepat — kunci dict adalah nama berkas persis seperti di
`data/raw/web/`, sama pola dengan `metadata_overrides` PDF di atas.

In [8]:
def guess_web_metadata(filename: str, meta: dict | None) -> dict:
    stem = Path(filename).stem
    year_match = re.search(r"20\d{2}", stem)
    year = int(year_match.group()) if year_match else pd.Timestamp.now().year
    slug = stem.split("__", 1)[-1]
    title = slug.replace("-", " ").strip() or slug
    return {"title": title, "program": "UNKNOWN", "year": year}


# EDIT di sini: kunci = nama berkas persis di data/raw/web/, nilai = override
# metadata (title/program/year) -- sama pola dengan metadata_overrides PDF.
web_metadata_overrides = {}

web_rows = []
for p in html_paths:
    snap_meta = ingest.read_web_snapshot_meta(p)
    guess = {**guess_web_metadata(p.name, snap_meta), **web_metadata_overrides.get(p.name, {})}
    web_rows.append(
        {
            "filename": p.name,
            "local_path": p,
            "sha256": html_hashes[p],
            "source_url": snap_meta["url"] if snap_meta else None,
            **guess,
        }
    )

web_meta_df = pd.DataFrame(
    web_rows, columns=["filename", "local_path", "sha256", "source_url", "title", "program", "year"]
)
web_meta_df

,filename,local_path,sha256,source_url,title,program,year


## Ekstraksi teks laman web (BeautifulSoup)

Laman web tidak berpaginasi seperti PDF — tiap laman dibungkus sebagai satu
"halaman" (`page=1`) supaya fungsi hilir yang sama (`write_interim_text`,
`is_extractable`, `build_document_record`) bisa dipakai ulang tanpa cabang kode
terpisah untuk web.

In [9]:
web_extracted = {}  # local_path -> list[{"page", "text", "tables", "method"}]
for row in web_meta_df.itertuples():
    pages = ingest.extract_text_html(row.local_path)
    web_extracted[row.local_path] = pages
    print(f"{row.filename}: {len(pages[0]['text'])} karakter teks")

## Rakit record dokumen & tulis teks interim

Dokumen dengan rata-rata < 20 karakter/halaman ditandai `extractable=false` (kemungkinan
besar gambar/poster) TETAPI tetap dicatat di `documents.jsonl`, bukan dibuang
(Batasan Masalah #2). PDF dan web memakai loop yang sama (`ingest.assemble_documents`)
supaya perakitan record tidak diduplikasi dua kali per sumber.

In [10]:
existing_ids: set[str] = set()

pdf_records, pdf_not_extractable = ingest.assemble_documents(meta_df, extracted, "pdf", existing_ids, "../data/interim")
web_records, web_not_extractable = ingest.assemble_documents(
    web_meta_df, web_extracted, "web", existing_ids, "../data/interim"
)

records = pdf_records + web_records
not_extractable = pdf_not_extractable + web_not_extractable

print(f"{len(records)} dokumen diproses ({len(pdf_records)} pdf + {len(web_records)} web), {len(not_extractable)} ditandai extractable=false")
if not_extractable:
    print("Tidak extractable:", not_extractable)

1 dokumen diproses (1 pdf + 0 web), 0 ditandai extractable=false


## ASSERT

In [11]:
assert len(records) == len(pdf_paths) + len(html_paths), "Jumlah record tidak sama dengan jumlah PDF+HTML sumber"

doc_ids = [r["doc_id"] for r in records]
assert len(doc_ids) == len(set(doc_ids)), "doc_id tidak unik"

for r in records:
    assert r["sha256"], f"sha256 kosong untuk {r['doc_id']}"
    assert Path("..", r["local_path"]).exists(), f"local_path tidak ada: {r['local_path']}"
    assert r["n_pages"] > 0, f"n_pages nol untuk {r['doc_id']}"

print("Semua assert lolos.")

Semua assert lolos.


## Tulis `documents.jsonl`

In [12]:
out_path = "../data/processed/documents.jsonl"
ingest.write_documents_jsonl(records, out_path)
print(f"Ditulis {len(records)} record ke {out_path}")

Ditulis 1 record ke ../data/processed/documents.jsonl


## Ringkasan

In [13]:
summary_df = pd.DataFrame(records)[
    ["doc_id", "title", "source_type", "program", "year", "n_pages", "extractable"]
]
print(f"Total dokumen: {len(summary_df)} ({summary_df['source_type'].value_counts().to_dict()})")
print(f"Total halaman: {summary_df['n_pages'].sum()}")
print(f"Tidak extractable: {(~summary_df['extractable']).sum()}")
summary_df

Total dokumen: 1 ({'pdf': 1})
Total halaman: 99
Tidak extractable: 0


,doc_id,title,source_type,program,year,n_pages,extractable
0,lpmb-2026-panduan-usulan-pengmas-internal-univ...,Panduan Usulan Pengmas Internal Universitas Ai...,pdf,Pengmas Internal,2026,99,True
